___
# <center>Árvores de decisão, florestas aleatórias e o arquivo de predições</center>
___

## Aula 24

Na aula 23, a regressão logística previu a divergência com AUC perto de 0,70. Ela
supõe que cada variável soma um efeito fixo na chance. Hoje vemos dois métodos
que **não fazem essa suposição** e aprendem regras de outro jeito:

- a **árvore de decisão** faz perguntas em sequência ("é recurso do MP?", "é da
  câmara X?") e chega a uma folha com uma probabilidade. É fácil de ler e
  encontra combinações de variáveis sozinha, mas decora com facilidade;
- a **floresta aleatória** faz a média de centenas de árvores diferentes. Perde
  a leitura fácil, mas costuma errar menos em casos novos.

O caminho do notebook:

1. ajustar e **ler** uma árvore;
2. escolher a **profundidade** por validação cruzada;
3. ajustar a **floresta** e ver as variáveis mais importantes;
4. **comparar** os três modelos de forma justa;
5. gerar o **arquivo de predições**, a receita do Projeto 08.

___
<div id="indice"></div>

## Índice

- [Preparação: a mesma da aula 23](#base)
- [1. Uma árvore de decisão](#parte1)
- [2. Escolher a profundidade](#parte2)
- [3. Floresta aleatória](#parte3)
- [4. Comparar os modelos](#parte4)
- [5. O arquivo de predições: a receita do Projeto 08](#parte5)

___
<div id="base"></div>

# Preparação: a mesma da aula 23

Mesma base (acórdãos de recursos criminais do TJSP), mesma pergunta (**houve voto
divergente?**) e mesmos passos. As três células abaixo refazem a aula 23: os
pacotes (agora com a árvore e a floresta no fim), a base, o preparo e a
separação entre treino e teste. Se tiver dúvida sobre alguma linha, volte ao
notebook da aula 23, que explica cada uma.

In [ ]:
# pandas: tabelas (a base, os resultados)
import pandas as pd
# plotnine: gráficos, com a mesma gramática do ggplot
from plotnine import *

# scikit-learn (sklearn): o pacote de machine learning. Cada import é uma peça:
from sklearn.model_selection import train_test_split   # separa treino e teste
from sklearn.model_selection import StratifiedKFold     # divide o treino em partes
from sklearn.model_selection import cross_val_score     # a validação cruzada
from sklearn.model_selection import cross_val_predict   # probabilidades de validação
from sklearn.compose import make_column_transformer     # um preparo para cada tipo de coluna
from sklearn.pipeline import make_pipeline              # encadeia etapas num objeto só
from sklearn.impute import SimpleImputer                # preenche valores faltantes
from sklearn.preprocessing import OneHotEncoder         # categorias viram colunas 0/1
from sklearn.preprocessing import StandardScaler        # números ficam na mesma escala
from sklearn.linear_model import LogisticRegression     # a regressão logística
from sklearn.metrics import roc_auc_score               # a AUC
from sklearn.tree import DecisionTreeClassifier, plot_tree  # a árvore e o desenho dela
from sklearn.ensemble import RandomForestClassifier         # a floresta aleatória

In [ ]:
base = pd.read_csv("https://jtrecenti.github.io/cdad2-202662/ex08-competicao/dados/treino.csv")
base = base.dropna(subset=["unanimidade"])                        # 39 acórdãos sem a informação
base["divergiu"] = (base["unanimidade"] == "Maioria").astype(int)  # 1 se houve voto divergente
print(base.shape)
base.head(3)

In [ ]:
categoricas = ["assunto", "camara", "comarca", "polo_mp", "rel_faculdade", "rel_tipo", "rel_quinto"]
numericas = ["ano_publicacao", "tempo", "rel_idade", "rel_tempo_magistratura"]


def cria_preparo():
    """Transforma as colunas da base em números que o modelo consegue usar."""
    # colunas categóricas: o faltante vira a categoria "não informado", e cada
    # categoria vira uma coluna 0/1 (categorias com menos de 5 acórdãos se juntam)
    categorias = make_pipeline(
        SimpleImputer(strategy="constant", fill_value="não informado"),
        OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=False),
    )
    # colunas numéricas: o faltante vira a mediana, e todas ficam com média 0 e
    # desvio padrão 1, para que a penalidade trate todas do mesmo jeito
    numeros = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    preparo = make_column_transformer(
        (categorias, categoricas), (numeros, numericas), verbose_feature_names_out=False)
    return preparo.set_output(transform="pandas")   # devolve uma tabela com nomes

colunas = categoricas + numericas
X_treino, X_teste, y_treino, y_teste = train_test_split(
    base[colunas], base["divergiu"], test_size=0.25, stratify=base["divergiu"], random_state=23)
pastas = StratifiedKFold(5, shuffle=True, random_state=23)

print("treino:", len(X_treino), "| teste:", len(X_teste), "| divergências no treino:", y_treino.sum())

**Um detalhe sobre o preparo.** A árvore não precisaria da padronização: ela
compara cada número com um limiar, e a escala não muda a comparação. Usamos o
mesmo preparo para os três modelos por simplicidade, e porque assim a comparação
da parte 4 fica justa.

[Volta ao Índice](#indice)

___
<div id="parte1"></div>

# 1. Uma árvore de decisão

A árvore procura, entre todas as colunas, **a pergunta que melhor separa** os
acórdãos com e sem divergência. Depois repete a busca dentro de cada lado, e
assim por diante. O hiperparâmetro `max_depth` limita quantas perguntas seguidas
ela pode fazer.

Começamos com profundidade 2, para dar para ler. Complete e rode.

In [ ]:
arvore = make_pipeline(cria_preparo(), DecisionTreeClassifier(max_depth=____, random_state=1))
arvore.fit(X_treino, y_treino)

# o desenho: arvore[0] é o preparo e arvore[1] é a árvore
_ = plot_tree(arvore[1], feature_names=arvore[0].get_feature_names_out(),
              filled=True, impurity=False, proportion=True, fontsize=8)

**Como ler cada caixa.**

- A primeira linha é a **pergunta**. Para uma dummy, `polo_mp_Ativo <= 0.5` quer
  dizer "não é recurso do MP" (a dummy vale 0). Quem responde **sim** vai para a
  esquerda; **não**, para a direita.
- `samples` é a proporção dos acórdãos do treino que chegaram àquela caixa.
- `value` traz duas proporções: a de acórdãos **sem** divergência e a de acórdãos
  **com** divergência, naquela caixa.
- As caixas de baixo, sem pergunta, são as **folhas**. A probabilidade que a
  árvore dá a um acórdão novo é a proporção de divergência da folha em que ele
  cai.

**Pergunta 1.** Qual a primeira pergunta da árvore? Escreva, em linguagem
natural, a regra que leva à folha com a maior proporção de divergência, e diga
qual é essa proporção.

[Volta ao Índice](#indice)

___
<div id="parte2"></div>

# 2. Escolher a profundidade

A profundidade é o **hiperparâmetro de complexidade** da árvore, como o `C` era
o da logística. Cada nível a mais divide as folhas de novo, e as folhas ficam
com menos acórdãos. Com poucos acórdãos por folha, a árvore começa a decorar.

O laço abaixo calcula, para cada profundidade, a AUC no treino e a AUC na
validação cruzada. Complete a medida.

In [ ]:
linhas = []
for prof in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15]:
    modelo = make_pipeline(cria_preparo(), DecisionTreeClassifier(max_depth=prof, random_state=1))
    # na validação cruzada: o desempenho em acórdãos que a árvore não viu
    validacao = cross_val_score(modelo, X_treino, y_treino, cv=pastas, scoring="____").mean()
    # no próprio treino: o quanto a árvore se ajusta ao que já viu
    modelo.fit(X_treino, y_treino)
    treino = roc_auc_score(y_treino, modelo.predict_proba(X_treino)[:, 1])
    linhas.append({"profundidade": prof, "treino": round(treino, 3), "validacao": round(validacao, 3)})

por_prof = pd.DataFrame(linhas)
por_prof

O gráfico mostra as duas curvas juntas. O `melt` empilha as colunas `treino` e
`validacao` numa coluna só, que é o formato que o plotnine usa para colorir.

In [ ]:
(ggplot(por_prof.melt(id_vars="profundidade", var_name="base", value_name="auc"),
        aes(x="profundidade", y="auc", color="base"))
 + geom_line(size=1) + geom_point(size=3)
 + labs(x="profundidade máxima", y="AUC", color="")
 + theme_minimal())

**Pergunta 2.**

a) Qual a melhor profundidade pela validação cruzada?

b) O que acontece com a distância entre treino e validação quando a árvore
cresce? Como isso se liga ao juizado com 80 juízes da aula 22?

[Volta ao Índice](#indice)

___
<div id="parte3"></div>

# 3. Floresta aleatória

Uma árvore sozinha muda muito com pequenas mudanças nos dados. A floresta usa
isso a seu favor:

1. cada árvore é ajustada numa **amostra sorteada com reposição** dos acórdãos do
   treino;
2. em cada pergunta, a árvore só pode escolher entre **algumas colunas
   sorteadas**;
3. para um acórdão novo, cada árvore dá a sua probabilidade, e a floresta faz a
   **média**.

As árvores erram de jeitos diferentes, e na média os erros se compensam. Os
argumentos: `n_estimators` é o número de árvores; `min_samples_leaf=3` exige
pelo menos 3 acórdãos por folha; `n_jobs=-1` usa todos os processadores do
computador. Complete com 200 árvores. A célula leva cerca de um minuto.

In [ ]:
floresta = make_pipeline(
    cria_preparo(),
    RandomForestClassifier(n_estimators=____, min_samples_leaf=3, n_jobs=-1, random_state=1))
auc_floresta = cross_val_score(floresta, X_treino, y_treino, cv=pastas, scoring="roc_auc").mean()
print("AUC da floresta na validação cruzada:", round(auc_floresta, 3))

A floresta não tem uma regra para ler, mas mostra quais colunas mais ajudaram:
a **importância** soma, em todas as árvores, quanto cada coluna melhorou as
divisões. As importâncias somam 1.

In [ ]:
floresta.fit(X_treino, y_treino)
importancia = pd.Series(floresta[1].feature_importances_, index=floresta[0].get_feature_names_out())
importancia.sort_values(ascending=False).head(10).round(3)

**Pergunta 3.** Uma pessoa lê a lista e conclui: "o tempo até o julgamento
**causa** a divergência". Por que a importância não permite essa conclusão?
Pense na aula sobre variáveis de confusão.

[Volta ao Índice](#indice)

___
<div id="parte4"></div>

# 4. Comparar os modelos

Para a comparação ser justa, os três modelos usam o **mesmo treino**, as
**mesmas pastas** e a **mesma medida** (a AUC na validação cruzada). Preencha o
melhor `C` da aula 23 (0,3) e a melhor profundidade que você achou na parte 2.

In [ ]:
modelos = {
    "logística": make_pipeline(cria_preparo(), LogisticRegression(C=____, max_iter=3000)),
    "árvore": make_pipeline(cria_preparo(), DecisionTreeClassifier(max_depth=____, random_state=1)),
    "floresta": floresta,
}
comparacao = {nome: cross_val_score(m, X_treino, y_treino, cv=pastas, scoring="roc_auc").mean()
              for nome, m in modelos.items()}
pd.Series(comparacao).round(3)

**Pergunta 4.** Um tribunal quer usar o modelo para **explicar** aos gabinetes
por que um caso foi sinalizado. Uma defensoria só quer **ordenar** a fila de
recursos. Qual modelo você indicaria para cada um? Justifique.

[Volta ao Índice](#indice)

___
<div id="parte5"></div>

# 5. O arquivo de predições: a receita do Projeto 08

No Projeto 08, você recebe `teste.csv`: as mesmas colunas, **sem a resposta**. O
portal calcula a AUC das suas probabilidades com as respostas que só ele tem.
Como a nota é a AUC, não é preciso escolher corte: basta mandar as
probabilidades.

A receita tem três passos:

1. ajustar o modelo escolhido na **base inteira** (agora não há mais o que
   guardar: o teste de verdade é o do portal);
2. prever as probabilidades no `teste.csv`;
3. salvar um CSV com as colunas `id` e `prob`.

Aqui o alvo ainda é a divergência, só para treinar a receita. **Não suba este
arquivo no Projeto 08**: ele prevê outra coisa. Complete o método que dá as
probabilidades.

In [ ]:
teste = pd.read_csv("https://jtrecenti.github.io/cdad2-202662/ex08-competicao/dados/teste.csv")

modelo_final = modelos["floresta"]
modelo_final.fit(base[colunas], base["divergiu"])      # 1. ajusta na base inteira

predicoes = modelo_final.____(teste[colunas])[:, 1]     # 2. probabilidades no teste

saida = pd.DataFrame({"id": teste["id"], "prob": predicoes})
saida.to_csv("predicoes_exemplo.csv", index=False)     # 3. o arquivo
saida.head()

### Do notebook ao Projeto 08

1. **Troque o alvo:** use `base["negou"]` no lugar de `base["divergiu"]`, e não
   tire as linhas sem `unanimidade` (elas valem para o projeto).
2. **Cuidado com o vazamento:** antes de usar uma coluna, pergunte: no dia em que
   eu for prever, eu já sei isto? A coluna `unanimidade` sai no mesmo julgamento
   que `negou`, mas está no `teste.csv`. Decida com cuidado e justifique para
   você mesmo.
3. **Escolha pela validação cruzada:** compare modelos, colunas e hiperparâmetros
   com a AUC na validação cruzada do treino. Só depois use uma das 3 submissões.
4. **No Colab**, o arquivo baixa com
   `from google.colab import files; files.download("predicoes.csv")`.

**Pergunta 5.** O Projeto 08 limita as submissões a 3. O que aconteceria com a
sua AUC no teste, e com o significado dela, se as submissões fossem ilimitadas?

[Volta ao Índice](#indice)